# L06 Practice — Make Your HDB Model Better & More Trustworthy

**Module 3 · Machine Learning & GenAI — Coaching Add-on**

---

### The story so far
In L01–L05 you built a model that predicts an HDB flat's resale price from **3 things**: floor area, lease year, and floor level. You deployed it in Streamlit. 

Today's question: **Is it any good? And can we make it better?**

### Your mission (about 1 hour)
1. **Measure** how wrong your current model is, in plain dollars.
2. **Add features** and see the error shrink.
3. **Compare two models** and learn how to pick one honestly.

> 💡 **How to use this notebook:** Run each cell with `Shift + Enter`. Cells marked **🔧 YOUR TURN** have blanks (`____`) for you to fill in. Stuck? Open the **▸ Hint** below it. Full answers are at the very bottom.

**Scenario:** You're a junior analyst at a property agency. Your manager says: *"Nice predictor — but how much can I trust the number it gives a client?"* Let's find out.

In [4]:
# Setup — run this first (needs internet to download the data)
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

url = ("https://raw.githubusercontent.com/kohjiaxuan/"
       "Predicting-HDB-Price-with-Machine-Learning/master/"
       "resale-flat-prices-based-on-registration-date-from-jan-2017-onwards.csv")
data = pd.read_csv(url)
data["floor_level"] = data["storey_range"].str.split(" ").str[0].astype(float)
print("Rows of real HDB sales loaded:", len(data))
data.head()

Rows of real HDB sales loaded: 50432


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,floor_level
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,10.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,1.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,1.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,4.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,1.0


## Part A — How wrong is the current (3-feature) model?

We'll rebuild your L05 model and, this time, **score** it.

**Key idea — train/test split:** We teach the model on 80% of the flats, then quietly test it on the other 20% it has *never seen*. That's the honest way to check it — like setting exam questions a student hasn't already memorised.

**Key idea — MAE (Mean Absolute Error):** the average gap between the predicted price and the real price. If MAE = \$60,000, the model is *on average* \$60k off. Lower is better.

### ✋ Pause & Predict
Before running: with only 3 features, do you think the model will be off by closer to **\$20k**, **\$60k**, or **\$120k**? Write your guess, then run the cell.

In [2]:
# Part A — baseline model with the original 3 features
features_3 = ["floor_area_sqm", "lease_commence_date", "floor_level"]
X = data[features_3]
y = data["resale_price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = LinearRegression()
baseline.fit(X_train, y_train)
preds = baseline.predict(X_test)

mae_baseline = mean_absolute_error(y_test, preds)
print(f"Baseline (3 features): on average off by S${mae_baseline:,.0f}")

Baseline (3 features): on average off by S$78,025


**What do you notice?** That dollar figure is your starting score. Every change you make from here, you'll compare against it. Write it down.

## Part B — 🔧 YOUR TURN: Add more features

Right now the model knows nothing about **what kind of flat** it is or **where** it is. A 5-room in Bishan and a 2-room in Yishun of the same size get treated the same — no wonder it struggles.

We'll add two columns: `flat_type` and `town`. These are **labels**, not numbers, so we convert them into 0/1 columns with `pd.get_dummies()` (a flat either *is* "4 ROOM" or it isn't).

In [6]:
# Part B — add flat_type and town
numeric = ["floor_area_sqm", "lease_commence_date", "floor_level"]
category = ["flat_type", "town"]

# 🔧 YOUR TURN: build X_more by combining the numeric + category columns,
#    then turning the categories into 0/1 columns.
X_more = pd.get_dummies(data[numeric + category], columns=category)   # <- fill the blank
y = data["resale_price"]

X_train, X_test, y_train, y_test = train_test_split(X_more, y, test_size=0.2, random_state=42)

richer = LinearRegression()
richer.fit(X_train, y_train)
mae_richer = mean_absolute_error(y_test, richer.predict(X_test))

print(f"Baseline (3 features): S${mae_baseline:,.0f}")
print(f"Richer  (5 features): S${mae_richer:,.0f}")
print(f"Improvement: S${mae_baseline - mae_richer:,.0f} smaller error")

Baseline (3 features): S$78,025
Richer  (5 features): S$46,726
Improvement: S$31,299 smaller error


<details><summary>▸ Hint</summary>

`get_dummies` needs to know which columns are categories. You already stored them in the variable `category`. So the blank is just `category`.
</details>

**What do you notice?** Did the error drop? By how much? More relevant information usually = better predictions — but not always, which is what Part C explores.

## Part C — 🔧 YOUR TURN: Choose a model + check for overfitting

Linear Regression draws straight-line relationships. A **Random Forest** can capture curves and combinations (e.g. "top floor *in a central town* is worth extra"). Let's see if it does better.

**Watch out for overfitting:** a model can score brilliantly on flats it trained on but flop on new ones — like a student who memorised past papers but can't handle new questions. We catch this by comparing the **training score** with the **test score**. A big gap = memorising, not learning.

In [8]:
# Part C — compare Linear Regression vs Random Forest
# 🔧 YOUR TURN: create a Random Forest model in the blank.
forest = RandomForestRegressor(n_estimators=100, random_state=42)   # <- fill the blank
forest.fit(X_train, y_train)

# Test-set error (on unseen flats) vs training-set error (on flats it learned from)
mae_test  = mean_absolute_error(y_test,  forest.predict(X_test))
mae_train = mean_absolute_error(y_train, forest.predict(X_train))

print(f"Linear Regression  test error: S${mae_richer:,.0f}")
print(f"Random Forest      test error: S${mae_test:,.0f}")
print()
print(f"Random Forest  TRAIN error: S${mae_train:,.0f}")
print(f"Random Forest   TEST error: S${mae_test:,.0f}")
print(f"Gap (overfitting signal): S${mae_test - mae_train:,.0f}")

Linear Regression  test error: S$46,726
Random Forest      test error: S$25,075

Random Forest  TRAIN error: S$17,167
Random Forest   TEST error: S$25,075
Gap (overfitting signal): S$7,909


<details><summary>▸ Hint</summary>

We imported it at the top as `RandomForestRegressor`. Put that in the blank.
</details>

**What do you notice?**
- Which model has the lower **test** error? That's usually the one to ship.
- Is the Random Forest's **train** error much smaller than its **test** error? That gap is overfitting — it knows the training flats almost perfectly but is less sharp on new ones.

## 🏆 Challenge (if you have time)
Pick **one**:
1. Add a third feature of your choice (e.g. `remaining_lease` cleaned to a number) and see if the error drops further.
2. Try `RandomForestRegressor(n_estimators=300)` — does more trees help the test error, or just slow it down?
3. Find the single flat in the test set where the model was **most wrong**. What was special about it?

## Part D — 🔧 (Stretch) Stacking: can a *team* of models beat the best one?

Your learner asked a great question: can we combine Random Forest and Gradient Boosting?

The feasible way to do this is **stacking** — train several different models *side by side*, then let one small final model (the "manager") learn how to best **blend** their predictions. The idea: each model makes different mistakes, so a smart blend can beat any single one.

> ⚠️ **Common misconception:** you do **not** chain them so Random Forest "narrows down" and Gradient Boosting "fine-tunes" the same price. Each model already predicts the full price on its own — you blend them, you don't hand one's answer to the other. (The one model that *does* "fine-tune its own mistakes step by step" is Gradient Boosting itself, internally.)

**Analogy:**
- Random Forest = average 100 independent guesses.
- Gradient Boosting = each guesser fixes the previous one's mistakes, in sequence.
- **Stacking** = collect guesses from several experts, then a manager learns whose opinion to trust when.

Let's test whether the extra complexity is actually worth it for our HDB data.

### ✋ Pause & Predict
The stacked team uses *more* computation than a single model. Do you think it will **beat** the best single model's test error, or only **tie** it for a lot more effort? Write your guess, then run.

In [12]:
# Part D — stack Random Forest + Gradient Boosting, blended by a Linear Regression "manager"
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor

# Two different base models (they make different kinds of mistakes)
base_models = [
    ("random_forest", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("grad_boost",    GradientBoostingRegressor(random_state=42)),
]

# 🔧 YOUR TURN: the "manager" that learns how to blend the base models.
#    Use a simple LinearRegression() as the final_estimator.
stack = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression(),          # <- fill the blank
)
stack.fit(X_train, y_train)
mae_stack = mean_absolute_error(y_test, stack.predict(X_test))

# Also score Gradient Boosting on its own, for a fair comparison
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
mae_gb = mean_absolute_error(y_test, gb.predict(X_test))

print(f"Linear Regression (Part B): S${mae_richer:,.0f}")
print(f"Random Forest    (Part C): S${mae_test:,.0f}")
print(f"Gradient Boosting (alone) : S${mae_gb:,.0f}")
print(f"Stacked team             : S${mae_stack:,.0f}")

Linear Regression (Part B): S$46,726
Random Forest    (Part C): S$25,075
Gradient Boosting (alone) : S$42,155
Stacked team             : S$25,089


<details><summary>▸ Hint</summary>

The "manager" is just another model. We've used it all along: `LinearRegression()`. Put that in the blank.
</details>

**What do you notice?**
- Did the **stacked team** beat the best single model? By how many dollars?
- Was the improvement big enough to justify training *three* models instead of one?
- Often a single well-tuned Gradient Boosting model is within a whisker of the stack — which is why, in practice, you usually reach for stacking only when that last bit of accuracy really matters (it's a classic competition-winning trick, but frequently overkill).

## ✅ Wrap-up

| Idea | In one line |
|---|---|
| **MAE** | Average dollar error — your model's report card. |
| **Train/test split** | Test on unseen flats for an honest score. |
| **Adding features** | More relevant info usually shrinks the error. |
| **Model choice** | Pick the lowest **test** error, not training error. |
| **Overfitting** | Big train-vs-test gap = memorising, not learning. |

**Up next:** plug your better model into the upgraded `app.py` and redeploy in Streamlit.

---
## Sample solutions (look only after trying!)

**Part B blank:** `X_more = pd.get_dummies(data[numeric + category], columns=category)`

**Part C blank:** `forest = RandomForestRegressor(n_estimators=100, random_state=42)`


**Part D blank:** `final_estimator=LinearRegression()`